In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from pathlib import Path
import re
import cv2
import os
import torch
import torch.nn.functional as F
from PIL import Image
from matplotlib.patches import Rectangle

from torchvision.models import resnet18, ResNet18_Weights

from pathlib import Path
import re

In [ ]:
def check_drift(paths, roi=(290, 55, 600, 350), ref=None):
    x0, y0, x1, y1 = roi
    def E(p):
        g = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)[y0:y1, x0:x1].astype(np.float32)
        g = cv2.GaussianBlur(g, (5, 5), 1.5)
        m = cv2.magnitude(cv2.Sobel(g, cv2.CV_32F, 1, 0, 3),
                          cv2.Sobel(g, cv2.CV_32F, 0, 1, 3))
        return m / (m.max() + 1e-6)

    hann = cv2.createHanningWindow((x1 - x0, y1 - y0), cv2.CV_32F)
    e0 = E(ref or paths[0])
    rows = [(str(p), *cv2.phaseCorrelate(e0, E(p), hann)[0],
             cv2.phaseCorrelate(e0, E(p), hann)[1]) for p in paths]

    df = pd.DataFrame(rows, columns=["path", "dx", "dy", "resp"])
    df["mag"] = np.hypot(df.dx, df.dy)
    print(df.mag.describe(percentiles=[.5, .95, .99]).round(3))
    print(f"\n>1 px: {(df.mag > 1).mean():.1%}   >3 px: {(df.mag > 3).mean():.1%}"
          f"   resp<0.3: {(df.resp < 0.3).mean():.1%}")
    return df

def load_roi(path):
    im = cv2.imread(str(path), cv2.IMREAD_COLOR)
    x0, y0, x1, y1 = ROI
    return im[y0:y1, x0:x1]

In [ ]:
ROI     = (290, 55, 600, 350)     # x0,y0,x1,y1
IN_W, IN_H = 640, 576             # upscale ×2 (facet ~34 px, patrz niżej)
D       = 64                      # wymiar po losowej selekcji
CACHE   = Path("padim_cache"); CACHE.mkdir(exist_ok=True)
DEV     = "cuda" if torch.cuda.is_available() else "cpu"

MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD  = np.array([0.229, 0.224, 0.225], np.float32)

In [ ]:
images_webcam_dir_path = Path("/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/webcam_useful_image/webcam_useful_images")
img_webcam_path_list = []
for img_path in images_webcam_dir_path.glob("*2024-01*"):
    img_webcam_path_list.append(str(img_path))

In [ ]:
img_webcam_path_list[:3]

In [ ]:
df=check_drift(img_webcam_path_list)

In [ ]:
mask = np.zeros((295, 310), np.uint8)
cv2.ellipse(mask, (150, 148), (140, 140), 0, 0, 360, 255, -1)
mb = mask > 0                                  # bool — tego używaj do indeksowania

# sanity check
roi = load_roi(img_webcam_path_list[0])
plt.imshow(cv2.cvtColor(cv2.bitwise_and(roi, roi, mask=mask), cv2.COLOR_BGR2RGB))

In [ ]:
net = resnet18(weights=ResNet18_Weights.DEFAULT).eval().to(DEV)
_f = {}
for n in ("layer1", "layer2"):
    getattr(net, n).register_forward_hook(lambda m, i, o, k=n: _f.__setitem__(k, o))

DIM_IDX = torch.randperm(192, generator=torch.Generator().manual_seed(42))[:D]

@torch.no_grad()
def embed(path):
    roi = load_roi(path)
    x = cv2.resize(roi, (IN_W, IN_H), interpolation=cv2.INTER_CUBIC)
    x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
    x = torch.from_numpy((x - MEAN) / STD).permute(2, 0, 1)[None].to(DEV)
    net(x)
    a = _f["layer1"]
    b = F.interpolate(_f["layer2"], size=a.shape[-2:], mode="nearest")
    e = torch.cat([a, b], 1)[0][DIM_IDX]        # (D, 144, 160)
    return e.cpu().numpy()

In [ ]:
def cache_path(p): return CACHE / (Path(p).stem + ".npy")
images_webcam_dir_path = Path("/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/webcam/2024/good")
img_webcam_path_list = sorted(images_webcam_dir_path.glob("*.jpg"))
for p in img_webcam_path_list:
    if not cache_path(p).exists():
        np.save(cache_path(p), embed(p).astype(np.float16))

Hp, Wp = 144, 160
P = Hp * Wp     

In [ ]:
images_webcam_dir_path = Path("/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/webcam/2024/good")
train = sorted(images_webcam_dir_path.glob("*.jpg"))

# przebieg 1: średnia
mu = np.zeros((P, D), np.float32)
for p in train:
    mu += np.load(cache_path(p)).astype(np.float32).reshape(D, P).T
mu /= len(train)

# przebieg 2: kowariancja
cov = np.zeros((P, D, D), np.float32)
for p in train:
    e = np.load(cache_path(p)).astype(np.float32).reshape(D, P).T - mu
    cov += np.einsum('pd,pe->pde', e, e)
cov /= (len(train) - 1)
cov += 0.01 * np.eye(D, dtype=np.float32)

inv = np.linalg.inv(cov)
del cov
np.savez(CACHE / "model.npz", mu=mu, inv=inv, dim_idx=DIM_IDX.numpy())

In [ ]:
def score(path):
    e = embed(path).reshape(D, P).T
    d = e - mu
    m2 = np.einsum('pd,pde,pe->p', d, inv, d)
    s = np.sqrt(np.maximum(m2, 0)).reshape(Hp, Wp).astype(np.float32)
    s = cv2.resize(s, (310, 295), interpolation=cv2.INTER_LINEAR)
    s = cv2.GaussianBlur(s, (0, 0), 4.0)
    s[~mb] = 0
    return s

def show(path, smap, vmax=None):
    roi = cv2.cvtColor(load_roi(path), cv2.COLOR_BGR2RGB)
    vmax = vmax or np.percentile(smap[mb], 99.5)
    fig, ax = plt.subplots(1, 2, figsize=(13, 6))
    ax[0].imshow(roi); ax[0].set_title(Path(path).stem)
    ax[1].imshow(roi); ax[1].imshow(np.ma.masked_where(~mb, smap),
                                    cmap="jet", alpha=0.5, vmin=0, vmax=vmax)
    ax[1].set_title(f"max={smap[mb].max():.1f}  p99={np.percentile(smap[mb],99):.1f}")

s = score(img_webcam_path_list[0]); show(img_webcam_path_list[0], s)

In [ ]:
def show(path, smap, vmax=None, cmap="inferno", ax=None):
    roi = cv2.cvtColor(load_roi(path), cv2.COLOR_BGR2RGB)
    if vmax is None:                                    # nie `or` — patrz niżej
        vmax = float(np.percentile(smap[mb], 99.5))

    own = ax is None
    if own:
        fig, ax = plt.subplots(1, 2, figsize=(13, 5.6))
    fig = ax[0].figure

    ax[0].imshow(roi)
    ax[0].set_title(Path(path).stem, fontsize=10)

    ax[1].imshow(roi)
    over  = np.ma.masked_where(~mb, smap)
    alpha = np.clip(smap / max(vmax, 1e-6), 0, 1) * 0.85
    im = ax[1].imshow(over, cmap=cmap, vmin=0, vmax=vmax,
                      alpha=alpha, interpolation="nearest")
    ax[1].contour(mb, levels=[0.5], colors="w", linewidths=0.6)
    ax[1].set_title(f"max={smap[mb].max():.1f}   "
                    f"p99={np.percentile(smap[mb],99):.1f}", fontsize=10)

    cb = fig.colorbar(im, ax=ax[1], fraction=0.046, pad=0.02)
    cb.set_label("odległość Mahalanobisa", fontsize=9)
    cb.solids.set_alpha(1.0)                            # inaczej pasek jest przezroczysty

    for a in ax:
        a.set_xticks([]); a.set_yticks([])
    if own:
        fig.tight_layout()
    return fig

In [ ]:
test_path = "/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/webcam_useful_image/mirror_exchange/image_2024-05-07_1100.jpg"
test_path = "/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/webcam_useful_image/mirror_exchange/image_2024-05-06_1800.jpg"
s = score(test_path)
show(test_path, s)

In [ ]:
test_path = "/home/pgliwny/Praca/Computer_vision_for_MAGIC/data/webcam/2024/20240405_1630_la.jpg"
s = score(test_path)
show(test_path, s)